# Homework 2: What Predicts Post-College Earnings?

MSE 125 — Spring 2026

## Setup

This homework covers Lectures 4–6: regression, feature engineering, and
validation. You will work with College Scorecard data from the U.S.
Department of Education. Each row in `field_of_study` is one academic
program at one institution. `scorecard` contains institution-level
characteristics. You will join these tables and build models predicting
`EARN_MDN_4YR` — median earnings 4 years after graduation.

You are encouraged to use AI coding assistants (Claude Code, Copilot,
ChatGPT) throughout this homework.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression, Lasso
from sklearn.model_selection import cross_val_score, train_test_split
import warnings
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')

In [2]:
# Load College Scorecard data directly from the course textbook repository
DATA_URL = 'https://raw.githubusercontent.com/stanford-mse-125/book/main/data/college-scorecard'
scorecard = pd.read_csv(f'{DATA_URL}/scorecard.csv')
field_of_study = pd.read_csv(f'{DATA_URL}/field_of_study.csv')

# Some columns are stored as strings (e.g., 'PrivacySuppressed' instead of NaN).
# Convert numeric columns so they are ready for analysis.
field_of_study['EARN_MDN_4YR'] = pd.to_numeric(field_of_study['EARN_MDN_4YR'], errors='coerce')
field_of_study['EARN_MDN_1YR'] = pd.to_numeric(field_of_study['EARN_MDN_1YR'], errors='coerce')
for col in ['SAT_AVG', 'UGDS', 'PCTPELL', 'PCTFLOAN', 'CONTROL', 'UG25ABV',
            'MD_EARN_WNE_P10', 'RET_FT4']:
    scorecard[col] = pd.to_numeric(scorecard[col], errors='coerce')

print(f"scorecard: {scorecard.shape[0]:,} rows x {scorecard.shape[1]} columns")
print(f"field_of_study: {field_of_study.shape[0]:,} rows x {field_of_study.shape[1]} columns")

scorecard: 7,703 rows x 122 columns
field_of_study: 227,980 rows x 9 columns

### How to use this notebook

Write your answers in the cells marked “Your code here” or “Your answer
here” below each question. You can add more cells if you need them
(Insert → Code cell or Text cell). Run each code cell with Shift+Enter.

### Submission

Submit your completed Jupyter notebook (`.ipynb`) on Gradescope. Keep
all code cells and their output visible. Write interpretations in
markdown cells, not code comments.

## Problem 1: First model

You advise a state legislator who is drafting a higher-education
scholarship bill. Before she asks hard questions about field of study,
she wants a one-number answer: how tightly does a school’s selectivity
track what its graduates earn? Build the merged College Scorecard
dataset, fit a first-pass line from `SAT_AVG` to `EARN_MDN_4YR`, and
hand back a one-sentence briefing line she can drop into her opening
remarks.

**Part (a):** Filter `field_of_study` to bachelor’s degrees
(`CREDDESC == "Bachelor's Degree"`). Drop rows where `EARN_MDN_4YR` is
missing (the setup cell already converted non-numeric values to NaN).
Join the result to `scorecard` on `UNITID` to bring in institution
features — you only need a few columns from `scorecard` (at minimum
`UNITID`, `SAT_AVG`, `PCTPELL`, `UGDS`, `CONTROL`). Summarize the merged
dataset: how many rows, how many unique schools (`UNITID`), and how many
distinct fields (unique `CIPCODE`)?

In [3]:
# Your code here

*Your answer here.*

**Part (b):** Fit a simple linear regression predicting `EARN_MDN_4YR`
from `SAT_AVG` (drop rows with missing SAT for now). Report the slope,
intercept, and R². Write the one-sentence briefing line the legislator
will read aloud — it must contain a dollar number and a qualifier about
how much of the variation the line explains.

In [4]:
# Your code here

*Your answer here.*

**Part (c):** Plot earnings vs. SAT average with the regression line
overlaid. Does the relationship look linear? Given the R² from part (b),
is this model underfitting the data, overfitting, or neither? What
single change would most improve it?

In [5]:
# Your code here

*Your answer here.*

## Problem 2: Where you go vs. what you study

A state legislator asks: “Should we steer students toward high-earning
fields, or does the school they attend matter more?”

**Part (a):** Add institution features to your regression: `PCTPELL`
(fraction of students on Pell grants), `UGDS` (undergraduate
enrollment), and `CONTROL` (1 = public, 2 = private nonprofit, 3 =
private for-profit — one-hot encode with public as the reference level
using `pd.get_dummies(drop_first=True)`). Report the coefficients and
R². Which institution features have the largest effects? Interpret the
`PCTPELL` coefficient in one sentence.

In [6]:
# Your code here

*Your answer here.*

**Part (b):** Create a 2-digit field code by extracting the first two
characters of `CIPCODE`:
`df['CIP2'] = df['CIPCODE'].astype(str).str[:2]`. Add these as dummy
variables to your model (using `pd.get_dummies(drop_first=True)`).
Report R² with and without the field dummies. Write one paragraph
answering the legislator’s question: does field of study or institution
predict earnings more?

In [7]:
# Your code here

*Your answer here.*

**Part (c):** Identify the three highest-earning and three
lowest-earning field categories by their dummy coefficients. Look up
what CIP codes correspond to these fields (use the `CIPDESC` column). Do
the results match your intuition?

In [8]:
# Your code here

*Your answer here.*

## Problem 3: The chief of staff’s margin notes

The legislator’s chief of staff sends back your Problem 2 model with
three sharpie marks in the margin: *“fan pattern in the residuals?”*,
*“coefficients in dollars or percents?”*, *“what about the 22% of
programs missing SAT?”* Your job is to resolve each flag before the memo
goes to committee. For each part below, end with **one sentence the
chief of staff can paste into the margin** as the fix.

**Part (a): Diagnose the scale.** Fit the Problem 2 model (institution
features + 2-digit CIP dummies) with `np.log(EARN_MDN_4YR)` as the
target instead of raw earnings. Produce two residual-vs.-fitted plots
side by side: one for the level model (from Problem 2), one for the log
model. In which plot is the fan pattern most pronounced — where the
residual spread grows visibly with the fitted value? To compare the two
models fairly, compute R² for the log model in dollar terms:
back-transform its predictions with `np.exp()` and compute
`1 - sum((y - exp(yhat_log))^2) / sum((y - y.mean())^2)`. Compare this
to the level model’s R². Also compare `y.mean()` to
`np.exp(yhat_log).mean()` — are they the same? In one sentence, explain
why or why not. Margin line: which scale you are shipping, and why.

In [9]:
# Your code here

*Your answer here.*

**Part (b): Translate the units.** Interpret the `PCTPELL` coefficient
in the log model — what does a 10 percentage-point increase in the Pell
fraction correspond to in percentage terms? Margin line: a single
sentence giving the effect in *percent of earnings*, phrased so the
chief of staff can reuse it verbatim.

*Your answer here.*

**Part (c): Interaction.** Create a binary indicator `is_engineering`
equal to 1 when `CIP2 == '14'` (engineering) and 0 otherwise. Add both
`is_engineering` and an interaction term `SAT_AVG * is_engineering` to
your model. Report and interpret the interaction coefficient: does
institutional selectivity (SAT average) matter more for engineering
programs or for other fields?

In [10]:
# Your code here

*Your answer here.*

**Part (d): Whose programs did we just drop?** Roughly 22% of rows in
the merged dataset lack `SAT_AVG`. Compare the distribution of
`CONTROL`, `UGDS`, and `PCTPELL` for rows with vs. without `SAT_AVG` (a
summary table or side-by-side boxplots). Is the missingness MCAR, MAR,
or MNAR? Margin line: the chief of staff needs to know *whose* programs
vanished from the analysis. Name the group in one sentence and flag the
risk in a second sentence.

In [11]:
# Your code here

*Your answer here.*

## Problem 4: Recommend the shipping model

The College Scorecard product team can ship exactly **one** earnings
model on their public-facing website. They are deciding among three
candidates of escalating complexity and have asked you to recommend one
by Friday. Fit the three candidates, compare them honestly, and close
with a named recommendation the product lead can forward to engineering.

Use a clean dataset with no missing values in any of the features or the
target. If you haven’t already, drop rows missing `SAT_AVG`, `PCTPELL`,
or `UGDS`.

**Part (a):** Fit three models, all including `SAT_AVG`, `PCTPELL`,
`UGDS`, and `CONTROL` dummies as base features:

-   **Model A:** base + 2-digit CIP field dummies (~37 dummies)
-   **Model B:** base + 4-digit CIP field dummies (~300 dummies)
-   **Model C:** base + 4-digit CIP field dummies + `SAT_AVG` × field
    interactions (~600 features)

Fit all three models and report the number of features in each.

In [12]:
# Your code here

**Part (b):** Report train R² for all three models. Which fits the
training data best?

In [13]:
# Your code here

*Your answer here.*

**Part (c):** Run 5-fold cross-validation on all three models (use
`cross_val_score` with `scoring='r2'`). Report the mean CV R² for each.
Does the ranking change compared to training R²? Explain in 2–3
sentences why the gap between train and CV R² grows with model
complexity.

In [14]:
# Your code here

*Your answer here.*

**Part (d):** Make a grouped bar chart showing train R² and CV R² side
by side for the three models. Explain what a large gap between train and
CV R² tells you about the model.

In [15]:
# Your code here

*Your answer here.*

**Part (e): Regularization.** Fit a `Lasso` regression on the Model B
features (4-digit CIP dummies) with three values of the regularization
parameter: `alpha=1`, `alpha=100`, and `alpha=10000`. For each, report
the 5-fold CV R² and the number of coefficients that the Lasso set to
exactly zero. How does the best Lasso compare to the unregularized
linear regression from Part (c)? What does increasing `alpha` do to the
model?

In [16]:
# Your code here

*Your answer here.*

**Part (f): Recommend.** Produce a Python string variable
`shipping_memo` (≤ 4 sentences) addressed to the Scorecard product lead.
It must name (i) which of the four candidates you ship — A, B, C, or the
best Lasso — (ii) its CV R², (iii) what you give up vs. the next-best
candidate, and (iv) one risk you are accepting by shipping it. Defend
the pick in the memo itself, not in a separate answer.

In [17]:
shipping_memo = """Your memo here."""
print(shipping_memo)

Your memo here.

*Your answer here.*

## Problem 5: Stress-test the shipping model

Before the Scorecard product team ships your Problem 4 recommendation,
the lead engineer asks two questions: “What if we’d had less data?” and
“How do we know the number we’re publishing is honest?” In Problem 1,
the SAT-only model was underfit — both R² and the scatter plot told you
the model was too simple, and adding features in Problem 2 fixed it.
Here you’ll see the opposite problem.

**Part (a):** Take Model B from Problem 4 (institution features +
4-digit CIP dummies). For each training set size n in {500, 1000, 2000,
5000, full dataset}, draw a random sample (`random_state=42`), fit Model
B, and compute train R² and 5-fold CV R². Plot both curves against n (a
learning curve). At n = 500, describe the gap between train and CV R².
What happens as n grows? In one sentence, explain why adding more
training data reduces overfitting.

In [18]:
# Your code here

*Your answer here.*

**Part (b):** Three summer interns each estimated how well Model C from
Problem 4 predicts earnings. All three used the same `X` (institution
features + 4-digit CIP dummies + SAT×field interactions — 605 features)
and `y` (`EARN_MDN_4YR`), on the clean dataset from Problem 4. Their
code and results are below.

``` python
# --- Intern 1 ---
model = LinearRegression()
model.fit(X, y)
print(f"R² = {model.score(X, y):.3f}")          # R² = 0.759

# --- Intern 2 ---
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)
best_r2 = -np.inf
for alpha in [0.01, 0.1, 1, 10, 100, 1000]:
    m = Lasso(alpha=alpha).fit(X_train, y_train)
    r2 = m.score(X_test, y_test)
    if r2 > best_r2:
        best_r2 = r2
print(f"R² = {best_r2:.3f}")                    # R² = 0.733

# --- Intern 3 ---
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)
model = LinearRegression()
scores = cross_val_score(model, X_train, y_train, cv=5)
model.fit(X_train, y_train)
print(f"CV R² = {scores.mean():.3f}")           # CV R² = 0.705
print(f"Test R² = {model.score(X_test, y_test):.3f}")  # Test R² = 0.729
```

The product lead asks: “Which number goes on the website?” For each
intern, explain in one sentence what their R² actually measures and
whether you trust it. Then answer: if the team had 100 candidate models
instead of 6 Lasso alphas, which intern’s number would change the most,
and in which direction?

*Your answer here.*

**Part (c):** Produce a Python string `procedure_memo` (≤ 4 sentences)
addressed to the lead engineer. It must: (i) name the procedure you used
and the honest R², (ii) explain in plain language what the other two
numbers actually measure, and (iii) state one rule of thumb for avoiding
the wrong procedure in future projects.

In [19]:
procedure_memo = """Your memo here."""
print(procedure_memo)

Your memo here.

*Your answer here.*

## Problem 6: The legislator’s report

Read the following draft memo carefully, then answer the questions
below.

> **MEMORANDUM**
>
> **To:** State Higher Education Committee **From:** Data Analytics
> Office **Re:** Field-of-Study Earnings Analysis
>
> Using data from the U.S. Department of Education’s College Scorecard,
> we analyzed median earnings four years after graduation for over
> 27,000 bachelor’s degree programs across 2,000 institutions. After
> controlling for institutional characteristics (selectivity, size,
> public/private status, and student demographics), we find that **field
> of study is the dominant predictor of post-college earnings**,
> explaining far more variation than institutional characteristics
> alone.
>
> Graduates of engineering and computer science programs earn a median
> of approximately \$95,000–\$105,000 four years after graduation,
> compared to approximately \$40,000–\$45,000 for graduates of fine
> arts, drama, and music programs — a gap of over \$50,000 that persists
> after controlling for institutional quality.
>
> **Recommendation:** The state should reallocate scholarship funding
> away from low-earning fields toward STEM programs to maximize the
> taxpayer’s return on investment in higher education.

**Part (a):** The analysis controlled for institution features. But it
did not control for differences among the *students* who choose each
field. Name at least two confounders — characteristics of students, not
programs — that could explain part of the field-earnings gap without
field choice being the direct cause.

*Your answer here.*

**Part (b):** The memo uses median earnings *4 years after graduation*.
Name one way this time horizon could be misleading when comparing
fields.

*Your answer here.*

**Part (c): Rewrite the memo.** Produce a Python string variable
`revised_recommendation` (≤ 3 sentences) that the Data Analytics Office
could paste into the memo *in place of* the current “Recommendation”
paragraph. Your version must be defensible given the confounders you
named in (a) and the time-horizon issue in (b). It may endorse, soften,
reverse, or replace the original recommendation — but it has to be a
recommendation the committee can act on, not a hedge.

In [20]:
revised_recommendation = """Your revised recommendation here."""
print(revised_recommendation)

Your revised recommendation here.

*Your answer here — justify the choices in the rewrite, sentence by
sentence. There is no single right answer; we are grading the quality of
your reasoning and whether the rewrite is defensible given what you
found in (a) and (b).*